# Focus Coach — browser demo (Gradio)

Runs the Focus Coach pipeline (YOLO26n head + phone detection, rules, time smoothing, session report) in your browser, with no install.

**How to use:** Runtime → Change runtime type → **T4 GPU** (a CPU works but is slower) → edit only the config cell → **Run all** → open a link printed at the end **in a new tab** and allow the camera there.
The link is temporary and stops working when this Colab session ends. If the live webcam feed is laggy, use the **Video** tab.

> **Note:** this notebook has not been run on a real Colab runtime yet. The demo code itself was tested locally; the steps that only exist in Colab (Drive, file upload, share link) were not.

**Config.** The only cell you may need to edit: your repo, and where the trained model is.

In [ ]:
REPO_URL = "https://github.com/majallen3690/focus-coach"  # EDIT: the GitHub repo that holds this project
MODEL_PATH = "/content/drive/MyDrive/focus-coach/runs/yolo26n_fpi/weights/best.pt"  # where Phase 2 saved the model on Drive
USE_DRIVE = True  # False: upload best.pt by hand in the model cell instead

**Runtime check.** Prints the GPU name. A CPU is fine for the Video and Image tabs, but the live tab will be slower.

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. The demo works on a CPU but is slower. For a GPU: Runtime > Change runtime type > T4 GPU.")

**Get the code.** Clones your repo. If it is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN`
(key icon in the left bar → Add new secret → turn on *Notebook access*). The token needs read access to the repo.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/focus-coach")
assert "YOUR-USERNAME" not in REPO_URL, "Edit REPO_URL in the config cell first."

if not REPO.exists():
    url = REPO_URL
    try:
        from google.colab import userdata

        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:  # no GITHUB_TOKEN secret: assume the repo is public
        pass
    done = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
    if done.returncode:  # print git's message without the token that is inside the URL
        raise RuntimeError(done.stderr.replace(url, REPO_URL))
os.chdir(REPO)
print("repo ready at", REPO)

**Get the model.** Copies `best.pt` from Drive, or shows an upload button when `USE_DRIVE = False`. It is saved as `models/best.pt`.

In [ ]:
import shutil

(REPO / "models").mkdir(exist_ok=True)
BEST = REPO / "models" / "best.pt"
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    assert Path(MODEL_PATH).exists(), f"No model at {MODEL_PATH}. Fix MODEL_PATH in the config cell, or set USE_DRIVE = False."
    shutil.copy(MODEL_PATH, BEST)
else:
    from google.colab import files

    print("Choose your best.pt file:")
    uploaded = files.upload()
    assert uploaded, "No file was uploaded."
    shutil.move(next(iter(uploaded)), BEST)
print("model ready:", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")

**Install** the two packages Colab does not have (Ultralytics and Gradio, pinned). Colab already has numpy, pandas, OpenCV, matplotlib and torch, so they are left alone on purpose: reinstalling them breaks Colab's own packages. Red lines about other packages are harmless; if Colab asks to restart the runtime, restart and Run all again.

In [ ]:
!pip install -q ultralytics==8.4.190 gradio==6.31.0

**Smoke test.** Runs the model on one example photo and checks the answer. Stops here with a clear message if something is wrong, before the demo starts.

In [ ]:
import sys

sys.path.insert(0, str(REPO))
from app.pipeline import FramePipeline
from ml import rules
import cv2

cfg = rules.load_config()
pipeline = FramePipeline(cfg, weights=BEST)
example = REPO / "demo/examples/images/03_on_phone.jpg"
result = pipeline.process(cv2.imread(str(example)), "off")
for d in result.dets:
    print(f"  {d['cls']:5s} {d['conf']:.2f}")
assert result.state == rules.ON_PHONE, f"Smoke test failed: expected ON_PHONE for {example.name}, got {result.state}. Is best.pt the right model?"
print("Smoke test passed: the model sees the phone and the rules say", result.state)

**Launch.** Starts the demo and prints a public link (`lhr.life`) that anyone can open, on any device.
The link comes from a free localhost.run tunnel over SSH. Gradio's own share server often refuses connections, and a Cloudflare tunnel
holds back the live updates, so the live tab would never start.

Open the link in **one** tab, allow the camera, click **Record**, then press **Start**. This cell can be run again; it restarts the demo.

In [ ]:
import importlib
import re
import shutil
import subprocess
import time

from IPython.display import HTML, display

subprocess.run(["git", "-C", str(REPO), "pull", "-q"])  # get the latest demo code
import demo.gradio_app as gradio_app
importlib.reload(gradio_app)

for name in ("demo", "app", "tunnel"):  # running this cell again: stop the previous demo and tunnel first
    old = globals().get(name)
    if hasattr(old, "close"):
        old.close()
    if hasattr(old, "kill"):
        old.kill()

app = gradio_app.build_demo(model_path=BEST)
# share=False: the public link comes from the tunnel below. inline=False: the camera only works in its own tab
app.queue().launch(server_port=7860, share=False, inline=False, quiet=True, prevent_thread_lock=True)

if not shutil.which("ssh"):
    subprocess.run(["apt-get", "install", "-y", "-qq", "openssh-client"], capture_output=True)
log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=30",
                           "-R", "80:127.0.0.1:7860", "nokey@localhost.run"], stdin=subprocess.DEVNULL, stdout=log, stderr=log)
public_url = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.lhr\.life", open("/content/tunnel.log").read())
    if found:
        public_url = found.group(0)
        break
    time.sleep(1)

if public_url:
    display(HTML(f'<h2>Focus Coach is running: <a href="{public_url}" target="_blank">{public_url}</a></h2>'
                 "<p>Open it in one tab only, allow the camera, click <b>Record</b>, then press <b>Start</b>.</p>"))
else:
    print("The public link did not start (see /content/tunnel.log). Run this cell again.")
app.block_thread()